# 🍄 Training Run — Team MOGU
**DL2026 Final Project — Session 11 Milestone**

This notebook documents the training of our DQN baseline and PPO main model.
All seeds are pinned for reproducibility.

**Hardware**: NVIDIA RTX 4050 Laptop GPU, WSL2 Ubuntu 24  
**Seed**: 42  
**Training time**: ~3 hours total

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import torch
import numpy as np
import random

# Pin all seeds for reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Seed: {SEED}')

## 1. Environment Setup

In [ ]:
from src.wrappers import make_mario_env, make_eval_env
from src.utils import load_config

cfg = load_config('../configs/default.yaml')
print('Config loaded:')
print(f'  World-Stage  : {cfg["environment"]["world"]}-{cfg["environment"]["stage"]}')
print(f'  Action type  : {cfg["environment"]["action_type"]}')
print(f'  Frame skip   : {cfg["environment"]["frame_skip"]}')
print(f'  Frame stack  : {cfg["environment"]["frame_stack"]}')
print(f'  Shaped rewards: {cfg["environment"]["shaped_rewards"]}')

# Test env
env = make_mario_env()
obs, _ = env.reset()
print(f'\nObservation shape: {obs.shape}')
print(f'Action space: {env.action_space}')
env.close()

## 2. DQN Baseline Training

In [ ]:
# To train DQN baseline, run:
# python train.py --model dqn --run_name dqn_baseline

# Or inline:
import subprocess
print('DQN training command:')
print('  python train.py --model dqn --run_name dqn_baseline')
print()
print('Expected result: ~1038 mean reward after 500k steps')
print('Known limitation: DQN requires large replay buffer (>11GB RAM)')
print('Mitigation: reduced buffer to 20k steps, accepted lower performance')

## 3. PPO Main Model Training

In [ ]:
# To train PPO main model, run:
# python train.py --model ppo --run_name ppo_main

print('PPO training command:')
print('  python train.py --model ppo --run_name ppo_main')
print()
print('Hyperparameters (from configs/default.yaml):')
ppo_cfg = cfg['ppo']
for k, v in ppo_cfg.items():
    print(f'  {k}: {v}')

## 4. Training Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

with open('../results/metrics.json') as f:
    metrics = json.load(f)

models  = ['DQN', 'PPO']
rewards = [metrics[m]['mean_reward'] for m in models]
x_pos   = [metrics[m]['mean_x_position'] for m in models]
comps   = [metrics[m]['completions'] for m in models]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
colors = ['#E07B54', '#4A90D9']

for ax, values, title, target in zip(
    axes,
    [rewards, x_pos, comps],
    ['Mean Reward', 'Mean X-Position', 'Level Completions (/100)'],
    [1500, None, None]
):
    bars = ax.bar(models, values, color=colors, edgecolor='white')
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
                f'{val:.0f}', ha='center', fontweight='bold')
    if target:
        ax.axhline(target, linestyle='--', alpha=0.5, label=f'Target: {target}')
        ax.legend()
    ax.set_title(title, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('DQN Baseline vs PPO Main Model', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('../results/figures/training_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved to results/figures/training_comparison.png')